# Extract and explore results data {#ref_tutorials_extract_and_explore_results_data}

`MAPDL`{.interpreted-text role="bdg-mapdl"} `LS-DYNA`{.interpreted-text
role="bdg-lsdyna"} `FLUENT`{.interpreted-text role="bdg-fluent"}
`CFX`{.interpreted-text role="bdg-cfx"}

This tutorial shows how to extract and explore results data from a
result file.

When you extract a result from a result file, DPF stores it in a
`Field<ansys.dpf.core.field.Field>`{.interpreted-text role="class"}.
This `Field` contains the data of the result associated with it.

:::: note
::: title
Note
:::

When DPF-Core returns the `Field` object, what Python actually has is a
client-side representation of the `Field`, not the entirety of the
`Field` itself. This means all the data is stored within the DPF
service. When building your workflows, the most efficient way of
interacting with result data is to minimize the exchange of data between
Python and DPF, either by using operators or by accessing exclusively
the data that is needed.
::::


# Get the result file

Import a result file. For this tutorial, use one available in the
`examples<ansys.dpf.core.examples>`{.interpreted-text role="mod"}
module. For more information about how to import your own result file in
DPF, see the `ref_tutorials_import_result_file`{.interpreted-text
role="ref"} tutorial.

:::: tip
::: title
Tip
:::

If you need to evaluate many result operators against the same file,
consider using a `StreamsContainer` to reuse open file handles and avoid
redundant mesh loads. See
`ref_tutorials_import_data_streams_for_multiple_operators`{.interpreted-text
role="ref"}.
::::

Extract the displacement results. The displacement
`Result<ansys.dpf.core.results.Results>`{.interpreted-text role="class"}
object gives a
`FieldsContainer<ansys.dpf.core.fields_container.FieldsContainer>`{.interpreted-text
role="class"} when evaluated. Get a `Field` from this `FieldsContainer`.


In [ ]:
# Import the ``ansys.dpf.core`` module
from ansys.dpf import core as dpf

# Import the operators and examples module
from ansys.dpf.core import examples, operators as ops

# Define the result file path
result_file_path_1 = examples.download_transient_result()

# Create the model
model_1 = dpf.Model(data_sources=result_file_path_1)

# Extract the displacement results for the last time step
disp_results = model_1.results.displacement.on_last_time_freq.eval()

# Get the displacement field for the last time step
disp_field = disp_results[0]
print(disp_field)

# Extract all the data from a `Field`

Extract the entire data in a `Field` as an array or a list.

## Get the data as an array (Numpy array)


In [ ]:
data_array = disp_field.data
print("Displacement data as an array: ", "\n", data_array)

Note that this array is a genuine, local, numpy array (overloaded by the
DPFArray).


In [ ]:
print("Array type: ", type(data_array))

# Get the data as a list


In [ ]:
data_list = disp_field.data_as_list
print("Displacement data as a list: ", "\n", data_list)

# Extract specific data from a field

If you need to access data for specific entities (node, element, \...),
you can extract it with two approaches:

- Based on its index (position in the `Field`) using the
  `get_entity_data()<ansys.dpf.core.field.Field.get_entity_data>`{.interpreted-text
  role="func"} method
- Based on the entity id using the
  `get_entity_data_by_id()<ansys.dpf.core.field.Field.get_entity_data_by_id>`{.interpreted-text
  role="func"} method

The `Field` data is organized with respect to its scoping ids. Note that
the element with `id=533` would correspond to an `index=2` within the
`Field`.


In [ ]:
# Get the index of the entity with id=533
index_533_entity = disp_field.scoping.index(id=533)
print("Index entity id=533: ", index_533_entity)

Note that scoping IDs are not sequential. Get the id of the element at
index 533.


In [ ]:
id_533_entity = disp_field.scoping.id(index=533)
print("Id entity index=533: ", id_533_entity)

# Get the data by the entity index {#ref_extract_specific_data_by_index}


In [ ]:
data_3_entity = disp_field.get_entity_data(index=3)
print("Data entity index=3: ", data_3_entity)

# Get the data by the entity id {#ref_extract_specific_data_by_id}


In [ ]:
data_533_entity = disp_field.get_entity_data_by_id(id=533)
print("Data entity id=533: ", data_533_entity)

# Extract specific data using a loop over the array

While the methods above are acceptable when requesting data for a few
elements or nodes, they should not be used when looping over the entire
array. For efficiency, a `Field` data can be recovered locally before
sending a large number of requests.


In [ ]:
# Create a deep copy of the field that can be accessed and modified locally.
with disp_field.as_local_field() as f:
    for i in disp_field.scoping.ids[2:50]:
        f.get_entity_data_by_id(i)

print(f)